In [1]:
class LogFileIterator:

    def __init__(self, filepath):
        self.file = open(filepath, "r")

    def __iter__(self):
        return self

    def __next__(self):

        line = self.file.readline()

        if line == "":
            self.file.close()
            raise StopIteration

        return line.strip()

In [2]:
def stream_errors(filepath):

    for line in LogFileIterator(filepath):

        if "ERROR" in line or "CRITICAL" in line:
            yield line


def stream_with_count(generator):

    count = 0

    for line in generator:
        count += 1
        yield count, line


def get_error_type(line):

    parts = line.split("[")

    if len(parts) >= 3:
        error_part = parts[2]
        error_type = error_part.split("]")[0]
        return error_type

    return "Unknown"

In [3]:
def group_errors(filepath):

    counts = {}

    for line in stream_errors(filepath):

        error_type = get_error_type(line)

        if error_type not in counts:
            counts[error_type] = 0

        counts[error_type] += 1

    return counts

In [4]:
def main():

    filepath = "/content/server.log"

    print("Streaming Errors (first few lines)\n")


    for count, line in stream_with_count(stream_errors(filepath)):

        print("Error", count, ":", line)

        if count == 6:
            break


    print("\nErrors grouped by type\n")

    counts = group_errors(filepath)

    for error_type in counts:
        print(error_type, ":", counts[error_type])

In [5]:
main()

Streaming Errors (first few lines)

Error 1 : [2024-01-01 10:02:40] ERROR [TimeoutError] Connection to database timed out
Error 2 : [2024-01-01 10:04:05] CRITICAL [DiskError] Disk space exhausted on /dev/sda1
Error 3 : [2024-01-01 10:06:30] ERROR [ValueError] Invalid input received from client
Error 4 : [2024-01-01 10:08:12] CRITICAL [MemoryError] Application ran out of memory
Error 5 : [2024-01-01 10:09:50] ERROR [ConnectionError] Failed to reach external API
Error 6 : [2024-01-01 10:11:14] ERROR [TimeoutError] Payment service request timed out

Errors grouped by type

TimeoutError : 2
DiskError : 1
ValueError : 1
MemoryError : 1
ConnectionError : 1
